Goal: every signal that has a turning direction should agree on which is left and which is right:
* steering angle;
* gyro z (yaw rate);
* IMU accel y (lateral);
* centerline curvature;
* car heading vs centerline theta.

In [5]:
import plotly.graph_objects as go
from src.loader import load_centerline

cl = load_centerline()

fig = go.Figure()

# track, coloured by curvature (red/blue = opposite signs, white = straight)
fig.add_trace(go.Scatter(
    x=cl["x"], y=cl["y"],
    mode="markers",
    marker=dict(size=4, color=cl["curvature"], colorscale="RdBu", cmid=0,
                colorbar=dict(title="curvature")),
    customdata=cl[["s", "curvature"]],
    hovertemplate="s=%{customdata[0]:.0f} m<br>curvature=%{customdata[1]:.4f}<extra></extra>",
    name="centerline",
))

# start and direction of travel
for s_mark, label in [(0, "s=0 (start)"), (20, "s=20")]:
    p = cl[cl["s"] == s_mark].iloc[0]
    fig.add_trace(go.Scatter(
        x=[p["x"]], y=[p["y"]], mode="markers+text",
        marker=dict(size=12, color="black"),
        text=[label], textposition="top right", name=label,
    ))

# equal scales so the track shape isn't distorted
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_layout(title="Centerline: direction of travel and curvature",
                  xaxis_title="x (m, east)", yaxis_title="y (m, north)",
                  height=700, width=1000)
fig.update_layout(legend=dict(orientation="h", y=-0.1))
fig.show()

centerline convention is confirmed: positive curvature = left turn

In [23]:
#hairpin coordinates from the graph above/
s_start = 708
s_end = 722

In [25]:
from src.loader import load
df = load("vehicle_curvilinear_coordinates")
mask1 = (df["s"] >=s_start) & (df["s"] <=s_end) # hairpin
mask2 = (df["t_s"] >=300) & (df["t_s"] <=360) # first lap
hairpin = df[mask1 & mask2]["t_s"]

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

t0 = hairpin.min() - 1
t1 = hairpin.max() + 1

signals = [
    ("steer_angle", "angle", "steering (deg)"),
    ("imu_angular_rate", "z", "yaw rate (deg/s)"),
    ("imu_acceleration", "y", "lateral acc (g)"),
]

fig = make_subplots(rows=len(signals), cols=1, shared_xaxes=True,
                    subplot_titles=[label for _, _, label in signals])

for i, (name, col, label) in enumerate(signals, start=1):
    df = load(name)
    w = df[(df["t_s"] >= t0) & (df["t_s"] <= t1)]
    fig.add_trace(go.Scatter(x=w["t_s"], y=w[col], name=label), row=i, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=i, col=1)  

fig.update_layout(height=500, width = 1000, title="Signals through the left hairpin",
                  showlegend=False)
fig.update_xaxes(title_text="t_s (s)", row=len(signals), col=1)
fig.show()

steering, yaw rate, and lateral acceleration all agree that going left is + and going right is -

finally, we check the car's headig which is wrapped.
we take it, we unwrap it, take its rate of change over time, convert it to deg/s

In [ ]:
import numpy as np
import plotly.graph_objects as go

pos = load("vehicle_position")
pos = pos[(pos["t_s"] >= t0) & (pos["t_s"] <= t1)]

# remove the +/-pi jumps, then rate of change in deg/s
heading = np.unwrap(pos["heading"].to_numpy())
heading_rate = np.degrees(np.gradient(heading, pos["t_s"].to_numpy()))

gyro = load("imu_angular_rate")
gyro = gyro[(gyro["t_s"] >= t0) & (gyro["t_s"] <= t1)]

fig = go.Figure()
fig.add_trace(go.Scatter(x=gyro["t_s"], y=gyro["z"], name="gyro z (deg/s)"))
fig.add_trace(go.Scatter(x=pos["t_s"], y=heading_rate, name="d(heading)/dt (deg/s)"))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(title="Yaw rate: gyro vs estimator heading",
                  xaxis_title="t_s (s)", yaxis_title="deg/s", height=450)
fig.show()

breakdown:
* we already knew the gyro's yaw rate is positive in a left turn from the hairpin.
* if the heading follows the same convention, then turning left should make the heading increase.
* so we unwrapped the heading, measured how fast it changes, and put it on top of the gyro's yaw rate.
8 they overlapped: both positive in the left turn and both negative in the right. So left = positive for the heading too.